# Kaggle Geospatial certificate CRS course in R

In [ ]:
library(tidyverse)
library(sf)
library(here)
library(units)

In [ ]:
regions <- read_sf(here("data/ghana/ghana/Regions/Map_of_Regions_in_Ghana.shp"))

In [ ]:
st_crs(regions)

In [ ]:
# Create a DataFrame with health facilities in Ghana
facilities_df <- read_csv(here("data/ghana/ghana/health_facilities.csv"))

There are a lot of missing lat/long values in the CSV that we need to drop.
`geopandas` just ignores them, but we need to filter them out for `sf`

In [ ]:
facilities_df %>% 
    summarise(across(c("Longitude", "Latitude"), ~ sum(is.na(.))))

In [ ]:
# Convert to sf object
# CRS 4326 = WGS84 (standard GPS coordinates)
facilities <- facilities_df %>% 
    filter(!is.na(Longitude), !is.na(Latitude)) %>% 
    st_as_sf(
        coords = c("Longitude", "Latitude"), # order: x = lon, y = lat
        crs = 4326,
        remove = FALSE  # Keep original lat/lon columns
    )

## Plots

In [ ]:
ggplot() +
    geom_sf(data = regions,
            linetype = "dotted") +
    geom_sf(data = st_transform(facilities, 32630),
            color = "blue",
            size = 1) +
    theme_minimal()

st_transform using **proj4** string

In [ ]:
st_transform(regions, "+proj=longlat +ellps=WGS84 +datum=WGS84 +no_defs")

## geometry attributes

In [ ]:
st_coordinates(head(st_geometry(facilities)))[,"X"]

In [ ]:
regions <- regions %>% 
    mutate(AREA = set_units(st_area(st_geometry(regions)), km^2))

In [ ]:
print(str_glue("The area of Ghana is {sum(regions$AREA)} square kilometers"))

# Exercises

In [ ]:
birds_df <- read_csv(here("data/purple_martin.csv"))

In [ ]:
colnames(birds_df) <- gsub(
    "\\.",
    "_",
    make.names(colnames(birds_df))
    )
unique_birds <- n_distinct(birds_df$tag_local_identifier)
print(str_glue("There are {unique_birds} different birds in the dataset."))

In [ ]:
birds <- birds_df %>% 
    st_as_sf(
        coords = c("location_long", "location_lat"),
        crs = 4326,
        remove = FALSE
    )

## Plot data

In [ ]:
world <- read_sf(here("data/tmap_World/tmap_World.shp"))

In [ ]:
americas <- world %>% 
    filter(continent %in% c("North America", "South America"))

In [ ]:
ggplot() +
    geom_sf(data = americas, fill = NA, color = "grey") +
    geom_sf(data = birds, size = 1, color = "blue") +
    theme_minimal()

In [ ]:
birds

### creates linestrings for the paths of each bird and the starting and ending point for each bird's journey

In [ ]:
path_gdf <- birds %>% 
    arrange(tag_local_identifier, timestamp) %>% # ensures points are ordered by timestamp
    summarize(
        .by = tag_local_identifier,
        # converts the MULTIPOINT obj from st_combine into a LINESTRING
        geometry = st_combine(geometry) %>% st_cast("LINESTRING")
    ) %>% 
    st_as_sf()
    
head(path_gdf)

In [ ]:
start_gdf <- birds %>% 
    arrange(tag_local_identifier, timestamp) %>% # ensures points are ordered by timestamp
    summarize(
        .by = tag_local_identifier,
        geometry = geometry[1]
    ) %>% 
    st_as_sf()

head(start_gdf$geometry %>% st_coordinates())

In [ ]:
end_gdf <- birds %>% 
    arrange(tag_local_identifier, timestamp) %>% # ensures points are ordered by timestamp
    summarize(
        .by = tag_local_identifier,
        geometry = geometry[length(geometry)]
    ) %>% 
    st_as_sf()

head(end_gdf$geometry %>% st_coordinates())

In [ ]:
head(end_gdf)

In [ ]:
ggplot() +
    geom_sf(data = americas, fill = NA, color = "grey") +
    geom_sf(data = start_gdf, aes(color = factor(tag_local_identifier)), size = 2) +
    geom_sf(data = path_gdf, aes(color = factor(tag_local_identifier)), linewidth = 1) +
    geom_sf(data = end_gdf, aes(color = factor(tag_local_identifier)), size = 2) +
    scale_color_manual(values = 
                           c("red",
                             'orange',
                             'gold',
                             'green',
                             'blue',
                             'purple',
                             'salmon',
                             'cyan',
                             'burlywood',
                             'cadetblue',
                             'maroon'
                             )
                       ) +
    theme_minimal()

## Protected areas calculations

In [ ]:
protected_filepath <- here("data/SAPA_Aug2019-shapefile/SAPA_Aug2019-shapefile/SAPA_Aug2019-shapefile-polygons.shp")
protected_areas <- read_sf(protected_filepath)

### shapefile is big 

rewrite as geojson for compression with xz

In [ ]:
st_write(protected_areas,
         here("data/SAPA_Aug2019-shapefile/SAPA_Aug2019-shapefile/SAPA_Aug2019-shapefile-polygons.geojson"))

FlatGeobuf with zstd compresses about as well as GeoJson, but the unpacked file is much smaller

In [ ]:
st_write(protected_areas,
         here("data/SAPA_Aug2019-shapefile/SAPA_Aug2019-shapefile/SAPA_Aug2019-flatgeobuf-polygons.fgb"))

In [ ]:
head(protected_areas)

In [ ]:
south_america <- americas %>% 
    filter(continent == "South America")

In [ ]:
ggplot() +
    geom_sf(data = south_america, fill = NA, color = "grey") +
    geom_sf(data = protected_areas, fill = "blue", color = "blue")

In [ ]:
P_Area <- sum(protected_areas$REP_AREA - protected_areas$REP_M_AREA)
print(str_glue("South America has {P_Area} square kilometers of protected areas."))

In [ ]:
head(south_america)

In [ ]:
st_crs(south_america)

In [ ]:
south_america <- st_transform(south_america, 3035)

In [ ]:
totalArea <- 
    set_units(
        sum(
            st_area(
                st_geometry(
                    st_transform(
                        south_america, 
                        3035
                    )
                )
            )
        ),
        km^2
    )

print(totalArea)

In [ ]:
# What percentage of South America is protected?
percentage_protected <- round((P_Area/totalArea) * 100, 2)
print(str_glue("Approximately {percentage_protected}% of South America is protected."))

In [ ]:
nom_protected_areas <- protected_areas %>% 
    filter(MARINE != 2)

birds_sa <- birds %>% 
    filter((st_coordinates(st_geometry(.))[,"Y"]) < 0)

In [ ]:
birds %>% filter((st_coordinates(st_geometry(.))[,"Y"]) < 0)

In [ ]:
ggplot() +
    geom_sf(data = south_america, fill = "white", color = "grey") +
    geom_sf(data = nom_protected_areas, fill = "skyblue", color = NA, alpha = 0.4) +
    geom_sf(data = birds_sa, color = 'red', alpha = 0.6, size = 2) +
    theme_minimal()